# Molecular representation of processed spatial transcriptomic data

Read the processed cell-level AnnData, check its required fields, and obtain the molecular representation used for niche discovery in the next notebook. Existing scVI features are reused; scVI is trained only when those features are absent. Expression values and spatial coordinates are not normalized or transformed by this notebook.

## 1. Read the input data

Set the input file and output directory below. The input must contain unique cell and gene identifiers, `obs['batch']`, `obs['core_id']`, and physical XY coordinates in `obsm['spatial']`. If `obsm['X_scVI']` is absent, an explicit raw-count matrix in `layers['counts']` is required for training. All data in this workflow come from the supplied input file.

In [ ]:
import os
import sys
from pathlib import Path

import anndata as ad
import numpy as np
import scipy.sparse as sp
import torch

# Locate the repository when Jupyter starts at its root or in this notebook directory.
ROOT = Path.cwd().resolve()
while not (ROOT / "notebooks/01_domain_discovery").is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Start Jupyter inside the spaGC repository.")
    ROOT = ROOT.parent


if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RUN_ID = os.environ.get("SPAGC_RUN_ID", "domain_reference_v2")
INPUT_H5AD = ROOT / "data/processed/merged_samples.h5ad"
OUTPUT_DIR = ROOT / "outputs/domain_discovery" / RUN_ID
OUTPUT_H5AD = OUTPUT_DIR / "01_embedding.h5ad"
SCVI_MODEL_DIR = OUTPUT_DIR / "01_scvi_model"

BATCH_KEY = "batch"
N_LATENT = 30
MAX_EPOCHS = 200
SEED = 42
ACCELERATOR = "gpu" if torch.cuda.is_available() else "cpu"

In [ ]:
if not INPUT_H5AD.is_file():
    raise FileNotFoundError(f"Input data not found: {INPUT_H5AD}")
if OUTPUT_H5AD.exists():
    raise FileExistsError(
        f"Output already exists: {OUTPUT_H5AD}. Select a new output directory."
    )

data = ad.read_h5ad(INPUT_H5AD)
print(f"Loaded {data.n_obs:,} cells and {data.n_vars:,} genes from {INPUT_H5AD}")

## 2. Check the cell metadata and coordinates

Technical batches condition scVI, whereas tissue-core identifiers delimit the spatial graphs constructed in the next notebook. Check that both identifiers are present for every cell and that cell coordinates are finite two-dimensional values. This step does not change the input data.

In [ ]:
if data.n_obs == 0 or data.n_vars == 0:
    raise ValueError("The input expression matrix is empty.")
if not data.obs_names.is_unique or not data.var_names.is_unique:
    raise ValueError("Cell and gene identifiers must be unique.")

for key in (BATCH_KEY, "core_id"):
    if key not in data.obs:
        raise KeyError(f"Missing cell metadata: {key}")
    if data.obs[key].isna().any():
        raise ValueError(f"Cell metadata contains missing values: {key}")

if "spatial" not in data.obsm:
    raise KeyError("Missing physical XY coordinates in obsm['spatial'].")
xy = np.asarray(data.obsm["spatial"])
if xy.shape != (data.n_obs, 2) or not np.isfinite(xy).all():
    raise ValueError(
        "obsm['spatial'] must contain finite coordinates with shape (n_cells, 2)."
    )

print(
    f"Validated {data.obs['core_id'].nunique()} tissue cores and {data.obs[BATCH_KEY].nunique()} batches."
)

## 3. Obtain the scVI molecular representation

If `X_scVI` is already present, reuse it without retraining or changing its dimensions. Otherwise, fit scVI to the explicit raw-count layer with technical-batch conditioning and store the latent representation in `obsm['X_scVI']`. The latent dimension and training budget above apply only to a new fit. Save newly fitted model weights alongside the output data.

A newly fitted scVI space is not assumed to match the space used by an older niche-clustering model; downstream discovery must use compatible or newly fitted models.

In [ ]:
if "X_scVI" in data.obsm:
    embedding = np.asarray(data.obsm["X_scVI"])
    if (
        embedding.ndim != 2
        or embedding.shape[0] != data.n_obs
        or embedding.shape[1] == 0
    ):
        raise ValueError("X_scVI must have shape (n_cells, n_features).")
    if not np.isfinite(embedding).all():
        raise ValueError("X_scVI contains non-finite values.")

    if "spagc_embedding" not in data.uns:
        data.uns["spagc_embedding"] = {
            "source": "supplied_embedding",
            "latent_dim": embedding.shape[1],
        }
    print(f"Reusing the supplied {embedding.shape[1]}-dimensional scVI representation.")
else:
    if "counts" not in data.layers:
        raise KeyError("scVI training requires raw counts in layers['counts'].")
    counts = data.layers["counts"]
    values = counts.data if sp.issparse(counts) else np.asarray(counts).ravel()
    if (
        not np.isfinite(values).all()
        or (values < 0).any()
        or not np.allclose(values, np.rint(values))
    ):
        raise ValueError("Raw counts must be finite, nonnegative and integer-valued.")
    if SCVI_MODEL_DIR.exists():
        raise FileExistsError(f"Model directory already exists: {SCVI_MODEL_DIR}")

    import scvi

    scvi.settings.seed = SEED
    scvi.model.SCVI.setup_anndata(data, layer="counts", batch_key=BATCH_KEY)
    model = scvi.model.SCVI(data, n_latent=N_LATENT)
    model.train(max_epochs=MAX_EPOCHS, accelerator=ACCELERATOR, devices=1)
    data.obsm["X_scVI"] = model.get_latent_representation()

    SCVI_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
    model.save(str(SCVI_MODEL_DIR))
    data.uns["spagc_embedding"] = {
        "source": "newly_fitted_scvi",
        "batch_key": BATCH_KEY,
        "latent_dim": N_LATENT,
    }
    print(
        f"Fitted a {N_LATENT}-dimensional scVI representation; model saved to {SCVI_MODEL_DIR}."
    )

## 4. Save the processed data

Check the final representation and save the AnnData for `02_niche_discovery.ipynb`. The output retains the input expression, metadata and coordinates, with scVI features in `obsm['X_scVI']`. Existing output files are not overwritten.

In [ ]:
embedding = np.asarray(data.obsm["X_scVI"])
if embedding.ndim != 2 or embedding.shape[0] != data.n_obs or embedding.shape[1] == 0:
    raise ValueError("The final scVI representation has an invalid shape.")
if not np.isfinite(embedding).all():
    raise ValueError("The final scVI representation contains non-finite values.")
if OUTPUT_H5AD.exists():
    raise FileExistsError(f"Output already exists: {OUTPUT_H5AD}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
data.write_h5ad(OUTPUT_H5AD)
print(
    f"Saved {data.n_obs:,} cells with {embedding.shape[1]} scVI features to {OUTPUT_H5AD}"
)